In [1]:
import pandas as pd
from itertools import permutations

In [2]:
df_zones =pd.read_csv("zones.csv")
df_zones.set_index("id", inplace = True)

### Imputing Missing Values

In [3]:
display(df_zones)

,name,time_from_airport_raw,time_from_airport_minutes,is_airport,requires_ferry_transfer,is_active,region,latitude,longitude,geofence_radius_km,geometry_type
id,,,,,,,,,,,
1,Akumal,1 hour and 30 minutes,90,False,False,True,Cancun-Riviera Maya,20.429090,-87.297629,NaN,Point
2,Boca Paila,2 hours and 10 minutes,130,False,False,True,Cancun-Riviera Maya,20.128112,-87.464964,NaN,Point
3,Calica,1 hour and 25 minutes,85,False,False,True,Cancun-Riviera Maya,20.575292,-87.123826,NaN,Point
4,Cancun,40 minutes,40,True,False,True,Cancun-Riviera Maya,21.162088,-86.821580,15.0,DestinationArea
5,Chetumal,5 hours and 30 minutes,330,False,False,True,Cancun-Riviera Maya,18.500129,-88.301905,15.0,DestinationArea
6,Chichen Itza,2 hours and 30 minutes,150,False,False,True,Cancun-Riviera Maya,20.684476,-88.567117,15.0,DestinationArea
7,Chiquila,2 hours,120,False,False,True,Cancun-Riviera Maya,21.426230,-87.341234,15.0,DestinationArea
8,Cozumel,about 1 hour. We will transfer you to the ferr...,60,False,True,True,Cancun-Riviera Maya,20.622625,-87.068549,NaN,Point
9,Isla Mujeres,50 minutes. We will transfer you to the ferry ...,50,False,True,True,Cancun-Riviera Maya,NaN,NaN,NaN,NaN


In [4]:
df_clean_zones = df_zones.copy()

In [5]:
df_clean_zones.dropna(subset=["latitude","longitude"],inplace = True)

In [6]:
display(df_clean_zones)

,name,time_from_airport_raw,time_from_airport_minutes,is_airport,requires_ferry_transfer,is_active,region,latitude,longitude,geofence_radius_km,geometry_type
id,,,,,,,,,,,
1,Akumal,1 hour and 30 minutes,90,False,False,True,Cancun-Riviera Maya,20.429090,-87.297629,NaN,Point
2,Boca Paila,2 hours and 10 minutes,130,False,False,True,Cancun-Riviera Maya,20.128112,-87.464964,NaN,Point
3,Calica,1 hour and 25 minutes,85,False,False,True,Cancun-Riviera Maya,20.575292,-87.123826,NaN,Point
4,Cancun,40 minutes,40,True,False,True,Cancun-Riviera Maya,21.162088,-86.821580,15.0,DestinationArea
5,Chetumal,5 hours and 30 minutes,330,False,False,True,Cancun-Riviera Maya,18.500129,-88.301905,15.0,DestinationArea
6,Chichen Itza,2 hours and 30 minutes,150,False,False,True,Cancun-Riviera Maya,20.684476,-88.567117,15.0,DestinationArea
7,Chiquila,2 hours,120,False,False,True,Cancun-Riviera Maya,21.426230,-87.341234,15.0,DestinationArea
8,Cozumel,about 1 hour. We will transfer you to the ferr...,60,False,True,True,Cancun-Riviera Maya,20.622625,-87.068549,NaN,Point
10,Maroma Beach,45 minutes,45,False,False,True,Cancun-Riviera Maya,20.740367,-86.964700,NaN,Point


In [7]:
zones = df_clean_zones.index
zonePermutations = list(permutations(zones,2)) 

In [8]:
len(zonePermutations)

650

### Trip Baseline Creation

In [11]:
import requests
import pandas as pd

API_KEY = "YOUR_API"
URL = "https://routes.googleapis.com/distanceMatrix/v2:computeRouteMatrix"
HEADERS = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "originIndex,destinationIndex,duration,distanceMeters,status,condition",
}

def wp(lat, lng):
    return {"waypoint": {"location": {"latLng": {"latitude": lat, "longitude": lng}}}}

def matrix_chunk(origins, dests):
    """origins/dests: lists of (id, lat, lng). Returns list of row dicts."""
    body = {
        "origins": [wp(lat, lng) for _, lat, lng in origins],
        "destinations": [wp(lat, lng) for _, lat, lng in dests],
        "travelMode": "DRIVE",
        "routingPreference": "TRAFFIC_UNAWARE",
    }
    r = requests.post(URL, json=body, headers=HEADERS)
    if not r.ok:
        print(r.status_code, r.text)
        r.raise_for_status()
    r.raise_for_status()
    rows = []
    for el in r.json():
        o = origins[el["originIndex"]][0]
        d = dests[el["destinationIndex"]][0]
        if o == d:
            continue  # skip diagonal
        if el.get("condition") != "ROUTE_EXISTS":
            rows.append({"origin_id": o, "destination_id": d,
                         "baseline_seconds": None, "baseline_meters": None,
                         "condition": el.get("condition")})
            continue
        rows.append({
            "origin_id": o,
            "destination_id": d,
            "baseline_seconds": int(el["duration"].rstrip("s")),
            "baseline_meters": el["distanceMeters"],
            "condition": "ROUTE_EXISTS",
        })
    return rows

# build the point list from your dataframe (index = id)
pts = [(i, row.latitude, row.longitude) for i, row in df_clean_zones.iterrows()]

# chunk so origins_chunk * dests_chunk <= 625
CHUNK = 25
all_rows = []
for i in range(0, len(pts), CHUNK):
    for j in range(0, len(pts), CHUNK):
        all_rows += matrix_chunk(pts[i:i+CHUNK], pts[j:j+CHUNK])

result = pd.DataFrame(all_rows)
result["baseline_minutes"] = result["baseline_seconds"] / 60
result.to_csv("zone_baseline_matrix.csv", index=False)

In [12]:
result["condition"].unique()

<ArrowStringArray>
['ROUTE_EXISTS']
Length: 1, dtype: str